In [ ]:
# ========================================================== #
# 11. Batch Size Experimentation Loop
# Purpose: Dynamically test different batch sizes to evaluate
# training speed versus validation performance.
# ========================================================== #

# Define the range of batch sizes to test (sticking to standard powers of 2)
batch_sizes_to_test = [9, 16, 32]
batch_experiments_results = []

print("🚀 Starting Batch Size Optimization Experiment...\n")

for b_size in batch_sizes_to_test:
    print("-" * 50)
    print(f"Testing Batch Size: {b_size}")
    print("-" * 50)
    
    # 1. Clear out memory from the previous iteration
    keras.backend.clear_session()
    set_seeds(42)  # Keep weight initialization identical
    
    # 2. Recreate DataLoaders with the current batch size
    temp_train_loader = DataLoader(train_dataset, batch_size=b_size, shuffle=True)
    temp_val_loader = DataLoader(val_dataset, batch_size=b_size, shuffle=False)
    
    # 3. Rebuild and compile a fresh copy of the model architecture
    temp_model = build_model()
    temp_model.compile(
        optimizer='adam', 
        loss='sparse_categorical_crossentropy', 
        metrics=['accuracy']
    )
    
    # 4. Measure execution time across a short run (5 epochs is enough to see speed/loss trends)
    start_time = time.perf_counter()
    history = temp_model.fit(
        temp_train_loader, 
        validation_data=temp_val_loader, 
        epochs=20, 
        verbose=1
    )
    end_time = time.perf_counter()
    
    # 5. Calculate metrics
    total_time = end_time - start_time
    avg_time_per_epoch = total_time / 20
    final_val_acc = history.history['val_accuracy'][-1]
    final_val_loss = history.history['val_loss'][-1]
    
    # 6. Record findings
    batch_experiments_results.append({
        'Batch Size': b_size,
        'Total Time (20 Epochs)': f"{total_time:.2f}s",
        'Avg Time/Epoch': f"{avg_time_per_epoch:.2f}s",
        'Final Val Loss': f"{final_val_loss:.4f}",
        'Final Val Accuracy': f"{final_val_acc:.4f}"
    })

# --- Display Results ---
print("\n" + "="*60)
print("EXPERIMENT COMPLETE: Summary Table")
print("="*60)
results_df = pd.DataFrame(batch_experiments_results)
display(results_df)